### 01 — pybaseball_installation_check

Purpose: Confirm `pybaseball` can pull Statcast and Fangraph data end-to-end against this 

Caching is enabled and pointed at `../data/cache/` so re-running this notebook is instant after the first pull.

In [1]:
from pathlib import Path

import pandas as pd
import pybaseball
from pybaseball import cache, statcast

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

print("pybaseball:", pybaseball.__version__)
print("pandas    :", pd.__version__)
print("cache dir :", cache.config.cache_directory)

pybaseball: 2.2.7
pandas    : 2.2.3
cache dir : C:\Users\ihuang\Documents\python projects\MLB-Pitch-Predictor\data\cache


In [2]:
df = statcast("2024-06-15", "2024-06-15")
df.shape

This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:00<00:00, 15.28it/s]


(4145, 118)

In [3]:
print("rows, cols:", df.shape)
print("\ndtype counts:")
print(df.dtypes.value_counts())
df.head(3)

rows, cols: (4145, 118)

dtype counts:
Int64             59
Float64           42
object            16
datetime64[ns]     1
Name: count, dtype: int64


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
2195,SL,2024-06-15,90.0,2.77,6.13,"Banks, Tanner",682998,621383,field_out,hit_into_play,...,1,2.47,-0.2,-0.2,45.8,3.728685,5.508876,22.93471,38.355313,30.63759
2249,ST,2024-06-15,82.7,3.11,5.98,"Banks, Tanner",682998,621383,None,ball,...,1,3.76,-1.36,-1.36,42.5,<NA>,<NA>,<NA>,<NA>,<NA>
2336,SI,2024-06-15,93.6,2.72,6.24,"Banks, Tanner",682998,621383,None,blocked_ball,...,1,1.76,1.17,1.17,49.1,<NA>,<NA>,<NA>,<NA>,<NA>


In [4]:
print(f"{len(df.columns)} columns:")
for c in df.columns:
    print(f"  {c}")

118 columns:
  pitch_type
  game_date
  release_speed
  release_pos_x
  release_pos_z
  player_name
  batter
  pitcher
  events
  description
  spin_dir
  spin_rate_deprecated
  break_angle_deprecated
  break_length_deprecated
  zone
  des
  game_type
  stand
  p_throws
  home_team
  away_team
  type
  hit_location
  bb_type
  balls
  strikes
  game_year
  pfx_x
  pfx_z
  plate_x
  plate_z
  on_3b
  on_2b
  on_1b
  outs_when_up
  inning
  inning_topbot
  hc_x
  hc_y
  tfs_deprecated
  tfs_zulu_deprecated
  umpire
  sv_id
  vx0
  vy0
  vz0
  ax
  ay
  az
  sz_top
  sz_bot
  hit_distance_sc
  launch_speed
  launch_angle
  effective_speed
  release_spin_rate
  release_extension
  game_pk
  fielder_2
  fielder_3
  fielder_4
  fielder_5
  fielder_6
  fielder_7
  fielder_8
  fielder_9
  release_pos_y
  estimated_ba_using_speedangle
  estimated_woba_using_speedangle
  woba_value
  woba_denom
  babip_value
  iso_value
  launch_speed_angle
  at_bat_number
  pitch_number
  pitch_name
  home_scor

In [5]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].to_frame("pct_missing")

,pct_missing
spin_dir,1.000000
tfs_deprecated,1.000000
break_length_deprecated,1.000000
break_angle_deprecated,1.000000
sv_id,1.000000
...,...
az,0.002895
ay,0.002895
ax,0.002895
batter_days_until_next_game,0.002654


In [6]:
print("pitch_type missingness:", df["pitch_type"].isna().mean())
print("pitch_name missingness:", df["pitch_name"].isna().mean())
print("\npitch_type value counts:")
print(df["pitch_type"].value_counts(dropna=False))
print("\npitch_name value counts:")
print(df["pitch_name"].value_counts(dropna=False))

pitch_type missingness: 0.0028950542822677927
pitch_name missingness: 0.0028950542822677927

pitch_type value counts:
pitch_type
FF      1426
SI       609
SL       512
CH       422
CU       331
FC       318
ST       292
FS       166
KC        41
SV        16
None      12
Name: count, dtype: int64

pitch_name value counts:
pitch_name
4-Seam Fastball    1426
Sinker              609
Slider              512
Changeup            422
Curveball           331
Cutter              318
Sweeper             292
Split-Finger        166
Knuckle Curve        41
Slurve               16
None                 12
Name: count, dtype: int64


---
### Player-Specific Feature Enrichment

The goal is to enrich each pitch row with **prior-year** pitcher and batter stats so the model has player-identity signal beyond raw game-state context.

**Why prior year?**  
Stats from the *current* season aggregate information from pitches thrown after the one we're trying to predict, so we must use prior year data to prevent data leakage.

| Table | pybaseball function | Why Do I Care
|---|---| --- |
| Pitcher pitch usage | `statcast_pitcher_pitch_arsenal(year, arsenal_type="n_")` | pitcher tendency is likely a huge predictor of next pitch type
| Pitcher per-pitch outcomes | `statcast_pitcher_arsenal_stats(year)` | pitch count intuitively affects next pitch type
| Batter quality stats | `statcast_batter_expected_stats(year)` | batter strength could affect fastball pitch probability
| Batter v.s. pitch types | `statcast_batter_pitch_arsenal(year)` | some batters are historically strong/weak against certain pitches

In [7]:
# ── Pitcher pitch-usage % (prior year) ────────────────────────────────────────
from pybaseball import statcast_pitcher_pitch_arsenal, statcast_pitcher_arsenal_stats

SEASON = int(df["game_year"].max())   # e.g. 2024
PRIOR  = SEASON - 1                   # e.g. 2023  ← leak-free

pit_usage = statcast_pitcher_pitch_arsenal(PRIOR, arsenal_type="n_")
print("pit_usage shape:", pit_usage.shape)
print("\nColumns:", pit_usage.columns.tolist())
pit_usage.head(3)

pit_usage shape: (603, 12)

Columns: ['last_name, first_name', 'pitcher', 'n_ff', 'n_si', 'n_fc', 'n_sl', 'n_ch', 'n_cu', 'n_fs', 'n_kn', 'n_st', 'n_sv']


,"last_name, first_name",pitcher,n_ff,n_si,n_fc,n_sl,n_ch,n_cu,n_fs,n_kn,n_st,n_sv
0,"Cole, Gerrit",543037,52.9,0.1,7.1,20.7,7.1,12.1,NaN,NaN,NaN,NaN
1,"Cease, Dylan",656302,43.2,NaN,NaN,38.6,3.0,15.2,NaN,NaN,NaN,NaN
2,"Gallen, Zac",668678,49.7,0.0,9.6,4.0,13.9,22.7,NaN,NaN,NaN,NaN


In [ ]:
# Pitcher per-pitch outcome stats (prior year)
pit_outcomes = statcast_pitcher_arsenal_stats(PRIOR, minPA=25)
print("pit_outcomes shape:", pit_outcomes.shape)
print("\nColumns:", pit_outcomes.columns.tolist())
pit_outcomes.head(6)

pit_outcomes shape: (1881, 20)

Columns: ['last_name, first_name', 'player_id', 'team_name_alt', 'pitch_type', 'pitch_name', 'run_value_per_100', 'run_value', 'pitches', 'pitch_usage', 'pa', 'ba', 'slg', 'woba', 'whiff_percent', 'k_percent', 'put_away', 'est_ba', 'est_slg', 'est_woba', 'hard_hit_percent']


,"last_name, first_name",player_id,team_name_alt,pitch_type,pitch_name,run_value_per_100,run_value,pitches,pitch_usage,pa,ba,slg,woba,whiff_percent,k_percent,put_away,est_ba,est_slg,est_woba,hard_hit_percent
0,"Burnes, Corbin",669203,MIL,FC,Cutter,1.4,23,1706,55.4,455,0.209,0.340,0.298,22.7,19.8,22.1,0.243,0.393,0.325,34.1
1,"Javier, Cristian",664299,HOU,FF,4-Seam Fastball,0.1,2,1682,58.4,450,0.236,0.455,0.331,26.5,21.8,19.0,0.231,0.500,0.346,41.3
2,"Strider, Spencer",675911,ATL,FF,4-Seam Fastball,0.5,8,1826,58.9,435,0.256,0.431,0.339,28.7,28.5,22.3,0.237,0.424,0.319,40.7
3,"Webb, Logan",657277,SF,CH,Changeup,2.2,30,1324,41.6,433,0.225,0.309,0.254,22.8,21.2,19.8,0.234,0.325,0.258,44.0
4,"Steele, Justin",657006,CHC,FF,4-Seam Fastball,1.0,17,1682,62.8,431,0.274,0.391,0.328,20.4,19.0,19.8,0.276,0.409,0.327,35.9
5,"Cole, Gerrit",543037,NYY,FF,4-Seam Fastball,1.7,30,1736,52.9,427,0.202,0.333,0.276,23.0,28.3,21.6,0.220,0.406,0.304,46.1


In [ ]:
# batter expected stats (prior year)
from pybaseball import statcast_batter_expected_stats, statcast_batter_pitch_arsenal

bat_xstats = statcast_batter_expected_stats(PRIOR)
print("bat_xstats shape:", bat_xstats.shape)
print("\nColumns:", bat_xstats.columns.tolist())
bat_xstats.head(3)

bat_xstats shape: (258, 14)

Columns: ['last_name, first_name', 'player_id', 'year', 'pa', 'bip', 'ba', 'est_ba', 'est_ba_minus_ba_diff', 'slg', 'est_slg', 'est_slg_minus_slg_diff', 'woba', 'est_woba', 'est_woba_minus_woba_diff']


,"last_name, first_name",player_id,year,pa,bip,ba,est_ba,est_ba_minus_ba_diff,slg,est_slg,est_slg_minus_slg_diff,woba,est_woba,est_woba_minus_woba_diff
0,"Semien, Marcus",543760,2023,753,565,0.276,0.258,0.018,0.478,0.434,0.044,0.354,0.335,0.019
1,"Acuña Jr., Ronald",660670,2023,735,562,0.337,0.351,-0.014,0.596,0.668,-0.072,0.428,0.461,-0.033
2,"Freeman, Freddie",518692,2023,730,521,0.331,0.315,0.016,0.567,0.567,0.000,0.411,0.406,0.005


In [ ]:
# Batter pitch-type vulnerability (prior year)
bat_vs_pitch = statcast_batter_pitch_arsenal(PRIOR, minPA=25)
print("bat_vs_pitch shape:", bat_vs_pitch.shape)
print("\nColumns:", bat_vs_pitch.columns.tolist())
bat_vs_pitch.head(6)

bat_vs_pitch shape: (2378, 20)

Columns: ['last_name, first_name', 'player_id', 'team_name_alt', 'pitch_type', 'pitch_name', 'run_value_per_100', 'run_value', 'pitches', 'pitch_usage', 'pa', 'ba', 'slg', 'woba', 'whiff_percent', 'k_percent', 'put_away', 'est_ba', 'est_slg', 'est_woba', 'hard_hit_percent']


,"last_name, first_name",player_id,team_name_alt,pitch_type,pitch_name,run_value_per_100,run_value,pitches,pitch_usage,pa,ba,slg,woba,whiff_percent,k_percent,put_away,est_ba,est_slg,est_woba,hard_hit_percent
0,"Bregman, Alex",608324,HOU,FF,4-Seam Fastball,0.7,6,914,32.8,282,0.278,0.444,0.374,9.8,8.9,10.7,0.281,0.484,0.385,40.8
1,"Kwan, Steven",680757,CLE,FF,4-Seam Fastball,-0.1,-1,1199,42.2,270,0.296,0.383,0.361,7.2,8.5,7.9,0.293,0.373,0.345,16.7
2,"Crawford, J.P.",641487,SEA,FF,4-Seam Fastball,1.5,16,1019,38.8,265,0.313,0.517,0.434,13.9,14.7,14.5,0.300,0.467,0.401,40.2
3,"Semien, Marcus",543760,TEX,FF,4-Seam Fastball,-0.4,-3,889,31.8,261,0.232,0.438,0.348,10.1,9.2,11.4,0.263,0.496,0.369,43.6
4,"Lowe, Nathaniel",663993,TEX,FF,4-Seam Fastball,0.6,8,1233,40.5,261,0.295,0.470,0.391,21.4,27.2,19.3,0.267,0.406,0.357,50.7
5,"Nimmo, Brandon",607043,NYM,FF,4-Seam Fastball,0.2,2,1145,40.2,239,0.270,0.470,0.379,19.2,23.0,16.4,0.257,0.468,0.368,53.4


In [ ]:
# Join all prior-year stats to the Statcast frame ───────────────────────────
pit_usage_w = pit_usage.rename(columns={"pitcher_id": "pitcher"})
pit_out_id_cols   = ["player_id", "last_name", "first_name", "year"]
pit_out_stat_cols = [c for c in pit_outcomes.columns if c not in pit_out_id_cols + ["pitch_type", "pitch_name"]]

pit_outcomes_w = (
    pit_outcomes
    .pivot_table(
        index="player_id",
        columns="pitch_type",
        values=pit_out_stat_cols,
        aggfunc="first",
    )
)
# flatten multi-index
pit_outcomes_w.columns = [f"{stat}_{pt}" for stat, pt in pit_outcomes_w.columns]

pit_outcomes_w = pit_outcomes_w.reset_index().rename(columns={"player_id": "pitcher"})
bat_xstats_w = bat_xstats.rename(columns={"player_id": "batter"})
bat_vs_id_cols   = ["player_id", "last_name", "first_name", "year"]
bat_vs_stat_cols = [c for c in bat_vs_pitch.columns if c not in bat_vs_id_cols + ["pitch_type", "pitch_name"]]

bat_vs_pitch_w = (
    bat_vs_pitch
    .pivot_table(
        index="player_id",
        columns="pitch_type",
        values=bat_vs_stat_cols,
        aggfunc="first",
    )
)
bat_vs_pitch_w.columns = [f"bat_{stat}_{pt}" for stat, pt in bat_vs_pitch_w.columns]
bat_vs_pitch_w = bat_vs_pitch_w.reset_index().rename(columns={"player_id": "batter"})

# merge to pitch level
df_enr = (
    df
    .merge(pit_usage_w,    on="pitcher", how="left", suffixes=("", "_pu"))
    .merge(pit_outcomes_w, on="pitcher", how="left", suffixes=("", "_po"))
    .merge(bat_xstats_w,   on="batter",  how="left", suffixes=("", "_bx"))
    .merge(bat_vs_pitch_w, on="batter",  how="left", suffixes=("", "_bv"))
)

print("Original shape :", df.shape)
print("Enriched shape :", df_enr.shape)

n_ff_col   = "n_ff"   if "n_ff"   in df_enr.columns else None
xwoba_col  = "xwoba"  if "xwoba"  in df_enr.columns else None
if n_ff_col:
    print(f"Pitcher arsenal coverage (n_ff non-null): "
          f"{df_enr[n_ff_col].notna().mean():.1%}")
if xwoba_col:
    print(f"Batter xwOBA coverage (xwoba non-null)  : "
          f"{df_enr[xwoba_col].notna().mean():.1%}")

preview_cols = (
    ["pitcher", "p_throws"]
    + [c for c in ["n_ff", "n_si", "n_sl", "n_ch", "n_cu"] if c in df_enr.columns]
    + ["batter", "stand"]
    + [c for c in ["xwoba", "xba"] if c in df_enr.columns]
)
df_enr[preview_cols].head(5)

Original shape : (4145, 118)
Enriched shape : (4145, 448)
Pitcher arsenal coverage (n_ff non-null): 67.5%


,pitcher,p_throws,n_ff,n_si,n_sl,n_ch,n_cu,batter,stand
0,621383,L,35.3,2.9,30.4,16.5,NaN,682998,L
1,621383,L,35.3,2.9,30.4,16.5,NaN,682998,L
2,621383,L,35.3,2.9,30.4,16.5,NaN,682998,L
3,621383,L,35.3,2.9,30.4,16.5,NaN,672695,R
4,621383,L,35.3,2.9,30.4,16.5,NaN,672695,R
